# Solution 2 — Manual Graph Neural Network (GNN) Approach

**Core idea**: Build a spatial graph where geohashes are nodes connected by proximity (~1 km radius). Do 1-hop and 2-hop message passing to aggregate neighbor demand signals. These GNN features capture:
- Demand spillover: traffic at one location influences adjacent locations
- Sparse geohash rescue: geohashes with no day-49 early data borrow from neighbors
- Spatial demand surface: global spatial pattern at each time slot

No PyTorch/TF needed — implemented with scipy KDTree + numpy.

In [1]:
import os
os.environ['DYLD_LIBRARY_PATH'] = (
    '/Users/amanmish/Library/Python/3.13/lib/python/site-packages/sklearn/.dylibs'
)

import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')
from scipy.spatial import KDTree
import lightgbm as lgb
import xgboost as xgb
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import Ridge
from sklearn.model_selection import KFold
from sklearn.metrics import r2_score
from sklearn.ensemble import HistGradientBoostingRegressor, ExtraTreesRegressor

SEED = 42
np.random.seed(SEED)
print('Libraries loaded')

Libraries loaded


In [2]:
train_raw = pd.read_csv('train.csv')
test_raw  = pd.read_csv('test.csv')
train = train_raw.copy()
test  = test_raw.copy()
print(f'Train: {train.shape}  Test: {test.shape}')
print(train.head(3))

Train: (77299, 11)  Test: (41778, 10)
   Index geohash  day timestamp    demand     RoadType  NumberofLanes  \
0      0  qp02z1   48       0:0  0.048804          NaN              1   
1      1  qp02zt   48       0:0  0.118507  Residential              3   
2      2  qp08bj   48       0:0  0.027132  Residential              1   

  LargeVehicles Landmarks  Temperature Weather  
0   Not Allowed        No          NaN     NaN  
1       Allowed       Yes    31.104565   Sunny  
2   Not Allowed        No    25.919267   Sunny  


## Step 1 — Geohash Decode & Spatial Graph Construction

In [3]:
BASE32 = '0123456789bcdefghjkmnpqrstuvwxyz'
B32MAP = {c: i for i, c in enumerate(BASE32)}

def gh_decode(s):
    lr = [-90., 90.]; lonr = [-180., 180.]; il = True
    for c in s:
        d = B32MAP[c]
        for b in [4, 3, 2, 1, 0]:
            if il: m = (lonr[0]+lonr[1])/2; lonr[1 if (d>>b)&1 else 0] = m
            else:  m = (lr[0]+lr[1])/2;     lr[1 if (d>>b)&1 else 0]   = m
            il = not il
    return (lr[0]+lr[1])/2, (lonr[0]+lonr[1])/2

all_gh  = pd.Series(np.concatenate([train['geohash'].values, test['geohash'].values])).unique()
lat_map = {g: gh_decode(g)[0] for g in all_gh}
lon_map = {g: gh_decode(g)[1] for g in all_gh}

for df in [train, test]:
    df['lat'] = df['geohash'].map(lat_map)
    df['lon'] = df['geohash'].map(lon_map)

print(f'Unique geohashes: {len(all_gh)}')
print(f'Lat range: {min(lat_map.values()):.4f} — {max(lat_map.values()):.4f}')
print(f'Lon range: {min(lon_map.values()):.4f} — {max(lon_map.values()):.4f}')

Unique geohashes: 1259
Lat range: 5.2377 — 5.4849
Lon range: -90.9723 — -90.5878


In [4]:
# ── Build spatial graph via KDTree ────────────────────────────────────────
# 0.012 degrees ≈ 1.3 km — connects geohashes within ~1 km radius
GRAPH_RADIUS = 0.012

gh_list   = list(all_gh)
gh_idx    = {g: i for i, g in enumerate(gh_list)}
coords    = np.array([[lat_map[g], lon_map[g]] for g in gh_list])

tree = KDTree(coords)
neighbor_idxs = tree.query_ball_tree(tree, r=GRAPH_RADIUS)

adjacency = {}  # geohash -> list of neighbor geohashes (excluding self)
for i, gh in enumerate(gh_list):
    adjacency[gh] = [gh_list[j] for j in neighbor_idxs[i] if j != i]

n_neighbors = np.array([len(adjacency[g]) for g in gh_list])
print(f'Graph radius: {GRAPH_RADIUS} degrees (~1.3 km)')
print(f'Avg neighbors per geohash: {n_neighbors.mean():.1f}')
print(f'Max neighbors: {n_neighbors.max()}  Min: {n_neighbors.min()}')
print(f'Isolated nodes (0 neighbors): {(n_neighbors==0).sum()}')

Graph radius: 0.012 degrees (~1.3 km)
Avg neighbors per geohash: 5.4
Max neighbors: 6  Min: 1
Isolated nodes (0 neighbors): 0


## Step 2 — Base Feature Engineering

In [5]:
# ── Temporal + hierarchy ─────────────────────────────────────────────────
for df in [train, test]:
    p = df['timestamp'].str.split(':', expand=True).astype(int)
    df['hour'] = p[0]; df['minute'] = p[1]
    df['time_idx'] = df['hour'] * 4 + df['minute'] // 15
    df['hour_sin'] = np.sin(2*np.pi*df['hour']/24)
    df['hour_cos'] = np.cos(2*np.pi*df['hour']/24)
    df['minute_sin'] = np.sin(2*np.pi*df['time_idx']/96)
    df['minute_cos'] = np.cos(2*np.pi*df['time_idx']/96)
    df['gh4'] = df['geohash'].str[:4]
    df['gh5'] = df['geohash'].str[:5]
    df['gh6'] = df['geohash'].str[:6]

# ── Impute missing categoricals / temperature ─────────────────────────────
CAT_MODE   = {c: train[c].mode()[0] for c in ['RoadType','Weather']}
GLOBAL_TEMP = train['Temperature'].median()
for df in [train, test]:
    for c in ['RoadType','Weather']: df[c] = df[c].fillna(CAT_MODE[c])
    df['Temperature'] = (df['Temperature']
        .fillna(df.groupby(['geohash','hour'])['Temperature'].transform('median'))
        .fillna(GLOBAL_TEMP))

d48 = train[train['day']==48].copy()
GLOBAL_MEAN = d48['demand'].mean()
print('Base temporal features done.')

Base temporal features done.


In [6]:
# ── Target encoding (geohash hierarchy → day-48 mean demand) ─────────────
for gh_col, K in [('geohash', 10), ('gh5', 5), ('gh4', 3), ('gh6', 8)]:
    stats   = d48.groupby(gh_col)['demand'].agg(['mean','count'])
    te      = (stats['count']*stats['mean'] + K*GLOBAL_MEAN) / (stats['count'] + K)
    te_name = f'{gh_col}_te'
    for df in [train, test]:
        df[te_name] = df[gh_col].map(te).fillna(GLOBAL_MEAN)

print('Target encoding done.')

Target encoding done.


In [7]:
# ── Lag features from day-48 ─────────────────────────────────────────────
lag_map = d48.groupby(['geohash','time_idx'])['demand'].mean()
for df in [train, test]:
    df['lag_d48'] = df.set_index(['geohash','time_idx']).index.map(lag_map).values
train.loc[train['day']==48, 'lag_d48'] = np.nan  # no leakage

NB = {'nb_m4':-4,'nb_m2':-2,'nb_m1':-1,'nb_p1':1,'nb_p2':2,'nb_p4':4}
for col, off in NB.items():
    shifted = d48.copy(); shifted['time_idx'] = shifted['time_idx'] - off
    nb_map = shifted.groupby(['geohash','time_idx'])['demand'].mean()
    for df in [train, test]:
        df[col] = df.set_index(['geohash','time_idx']).index.map(nb_map).values

# ── Day-48 aggregate stats ────────────────────────────────────────────────
gh_mean48_map   = d48.groupby('geohash')['demand'].mean()
gh_max48_map    = d48.groupby('geohash')['demand'].max()
gh_std48_map    = d48.groupby('geohash')['demand'].std()
gh_med48_map    = d48.groupby('geohash')['demand'].median()
gh_q25_48_map   = d48.groupby('geohash')['demand'].quantile(0.25)
gh_q75_48_map   = d48.groupby('geohash')['demand'].quantile(0.75)
slot_mean48_map = d48.groupby('time_idx')['demand'].mean()
d48h = d48.copy(); d48h['hg'] = d48h['time_idx']//4
gh_hmean48_map  = d48h.groupby(['geohash','hg'])['demand'].mean()
d48['q6'] = d48['time_idx']//6
gh_q6_map = d48.groupby(['geohash','q6'])['demand'].mean()
d48['q2'] = d48['time_idx']//2
gh_q2_map = d48.groupby(['geohash','q2'])['demand'].mean()
gh_peak_tidx_map = (d48.loc[d48.groupby('geohash')['demand'].idxmax(),
                             ['geohash','time_idx']].set_index('geohash')['time_idx'])

for df in [train, test]:
    df['gh_mean48']  = df['geohash'].map(gh_mean48_map).fillna(GLOBAL_MEAN)
    df['gh_max48']   = df['geohash'].map(gh_max48_map).fillna(GLOBAL_MEAN)
    df['gh_std48']   = df['geohash'].map(gh_std48_map).fillna(0.)
    df['gh_med48']   = df['geohash'].map(gh_med48_map).fillna(GLOBAL_MEAN)
    df['gh_q25_48']  = df['geohash'].map(gh_q25_48_map).fillna(GLOBAL_MEAN)
    df['gh_q75_48']  = df['geohash'].map(gh_q75_48_map).fillna(GLOBAL_MEAN)
    df['slot_mean48']= df['time_idx'].map(slot_mean48_map).fillna(GLOBAL_MEAN)
    df['gh_hmean48'] = df.set_index(['geohash','hour']).index.map(gh_hmean48_map).values
    df['q6'] = df['time_idx']//6
    df['q2'] = df['time_idx']//2
    df['gh_q6mean48'] = df.set_index(['geohash','q6']).index.map(gh_q6_map).fillna(GLOBAL_MEAN).values
    df['gh_q2mean48'] = df.set_index(['geohash','q2']).index.map(gh_q2_map).fillna(GLOBAL_MEAN).values
    df['gh_d48_peak_tidx'] = df['geohash'].map(gh_peak_tidx_map).fillna(48)
    df['dist_to_peak'] = (df['time_idx'] - df['gh_d48_peak_tidx']).abs()

for df in [train, test]:
    df['lag_d48']    = df['lag_d48'].fillna(df['gh_hmean48']).fillna(df['slot_mean48']).fillna(GLOBAL_MEAN)
    df['gh_hmean48'] = df['gh_hmean48'].fillna(GLOBAL_MEAN)
    for col in NB: df[col] = df[col].fillna(df['lag_d48'])

print('Day-48 lag features done.')

Day-48 lag features done.


In [8]:
# ── Early day-49 features ─────────────────────────────────────────────────
d49_early = train[train['day']==49].copy()
d49_early['gh5'] = d49_early['geohash'].str[:5]
ts_d49 = d49_early['time_idx'].unique()
d49_mean_map  = d49_early.groupby('geohash')['demand'].mean()
d49_max_map   = d49_early.groupby('geohash')['demand'].max()
d49_std_map   = d49_early.groupby('geohash')['demand'].std()
d49_count_map = d49_early.groupby('geohash')['demand'].count()
d48_match_map = d48[d48['time_idx'].isin(ts_d49)].groupby('geohash')['demand'].mean()
last_tidx_d49 = d49_early['time_idx'].max()
d49_last_map  = d49_early[d49_early['time_idx']==last_tidx_d49].groupby('geohash')['demand'].mean()
d48_at_last   = d48[d48['time_idx']==last_tidx_d49].groupby('geohash')['demand'].mean()

d49_slope_data = []
for gh, grp in d49_early.groupby('geohash'):
    t = grp['time_idx'].values.astype(float); y_ = grp['demand'].values
    d49_slope_data.append((gh, np.polyfit(t, y_, 1)[0] if len(t)>1 else 0.0))
d49_slope_map = pd.Series({k:v for k,v in d49_slope_data})

GLOBAL_D49M       = d49_mean_map.mean()
GLOBAL_RATIO      = (d49_mean_map/(d48_match_map+1e-8)).clip(0.2,5.0).median()
GLOBAL_LAST_RATIO = (d49_last_map/(d48_at_last+1e-8)).clip(0.2,5.0).median()

d48m_gh5 = d48[d48['time_idx'].isin(ts_d49)].copy()
d48m_gh5['gh5'] = d48m_gh5['geohash'].str[:5]
gh5_d49_mean_map = d49_early.groupby('gh5')['demand'].mean()
gh5_d49_max_map  = d49_early.groupby('gh5')['demand'].max()
gh5_d48_mean_map = d48m_gh5.groupby('gh5')['demand'].mean()
gh5_ratio_map    = (gh5_d49_mean_map/(gh5_d48_mean_map+1e-8)).clip(0.2,5.0)

K_SMOOTH = 5
for df in [train, test]:
    gh5_col = df['geohash'].str[:5]
    n_obs   = df['geohash'].map(d49_count_map).fillna(0)
    d49m = df['geohash'].map(d49_mean_map); d49x = df['geohash'].map(d49_max_map)
    mask_m = d49m.isna(); mask_x = d49x.isna()
    d49m.loc[mask_m] = gh5_col.loc[mask_m].map(gh5_d49_mean_map)
    d49x.loc[mask_x] = gh5_col.loc[mask_x].map(gh5_d49_max_map)
    df['d49_mean']  = d49m.fillna(GLOBAL_D49M)
    df['d49_max']   = d49x.fillna(GLOBAL_D49M)
    df['d49_std']   = df['geohash'].map(d49_std_map).fillna(0.)
    df['d49_last']  = df['geohash'].map(d49_last_map).fillna(GLOBAL_D49M)
    df['d49_slope'] = df['geohash'].map(d49_slope_map).fillna(0.)
    last_r = df['geohash'].map(d49_last_map)/(df['geohash'].map(d48_at_last)+1e-8)
    df['d49_last_ratio'] = last_r.clip(0.2,5.0).fillna(GLOBAL_LAST_RATIO)
    num = df['geohash'].map(d49_mean_map)
    den = df['geohash'].map(d48_match_map).fillna(GLOBAL_MEAN)
    raw_ratio  = (num/(den+1e-8)).clip(0.2,5.0)
    cluster_r  = gh5_col.map(gh5_ratio_map).fillna(GLOBAL_RATIO)
    indiv_r    = raw_ratio.fillna(cluster_r)
    df['early_ratio']        = indiv_r.fillna(GLOBAL_RATIO)
    df['early_ratio_smooth'] = ((n_obs*indiv_r+K_SMOOTH*cluster_r)/(n_obs+K_SMOOTH)).fillna(GLOBAL_RATIO).clip(0.2,5.0)

day48_mask_fe = train['day']==48
train.loc[day48_mask_fe, ['d49_mean','d49_max','d49_std','d49_last','d49_slope',
                           'd49_last_ratio','early_ratio','early_ratio_smooth']] = np.nan
for df in [train, test]:
    for c in ['d49_mean','d49_max']: df[c] = df[c].fillna(GLOBAL_D49M)
    df['d49_std']             = df['d49_std'].fillna(0.)
    df['d49_last']            = df['d49_last'].fillna(GLOBAL_D49M)
    df['d49_slope']           = df['d49_slope'].fillna(0.)
    df['d49_last_ratio']      = df['d49_last_ratio'].fillna(GLOBAL_LAST_RATIO)
    df['early_ratio']         = df['early_ratio'].fillna(GLOBAL_RATIO)
    df['early_ratio_smooth']  = df['early_ratio_smooth'].fillna(GLOBAL_RATIO)
    df['adjusted_lag']        = df['lag_d48']*df['early_ratio']
    df['adjusted_lag_smooth'] = df['lag_d48']*df['early_ratio_smooth']
    df['lag_slope']           = df['nb_p2']-df['nb_m2']
    df['lag_slope_broad']     = df['nb_p4']-df['nb_m4']

# ── Weather features ──────────────────────────────────────────────────────
wx_map   = d48.groupby(['geohash','time_idx'])['Weather'].first()
temp_map = d48.groupby(['geohash','time_idx'])['Temperature'].mean()
for df in [train, test]:
    idx = df.set_index(['geohash','time_idx']).index
    df['weather_d48']     = idx.map(wx_map).fillna(CAT_MODE['Weather']).values
    df['temp_d48']        = idx.map(temp_map).fillna(GLOBAL_TEMP).values
    df['weather_changed'] = (df['Weather'] != df['weather_d48']).astype(int)
    df['temp_diff']       = df['Temperature'] - df['temp_d48']

wx_time_map = d48.groupby(['time_idx','Weather'])['demand'].mean()
gh_wx_map   = d48.groupby(['geohash','Weather'])['demand'].mean()
for df in [train, test]:
    idx_today = list(zip(df['time_idx'].values, df['Weather'].values))
    idx_yest  = list(zip(df['time_idx'].values, df['weather_d48'].values))
    wx_today = pd.Series(idx_today, index=df.index).map(wx_time_map).fillna(df['slot_mean48'])
    wx_yest  = pd.Series(idx_yest,  index=df.index).map(wx_time_map).fillna(df['slot_mean48'])
    df['wx_demand_ratio'] = (wx_today/(wx_yest+1e-8)).clip(0.2,5.0)
    df['wx_adj_lag']      = df['lag_d48']*df['wx_demand_ratio']
    df['full_adj_lag']    = df['adjusted_lag_smooth']*df['wx_demand_ratio']
    gh_wx_idx     = list(zip(df['geohash'].values, df['Weather'].values))
    gh_wx_d48_idx = list(zip(df['geohash'].values, df['weather_d48'].values))
    df['gh_wx_mean']     = pd.Series(gh_wx_idx,     index=df.index).map(gh_wx_map).fillna(df['gh_mean48']).values
    df['gh_wx_d48_mean'] = pd.Series(gh_wx_d48_idx, index=df.index).map(gh_wx_map).fillna(df['gh_mean48']).values
    df['gh_wx_ratio']    = (df['gh_wx_mean']/(df['gh_wx_d48_mean']+1e-8)).clip(0.2,5.0)

for df in [train, test]:
    df['adj_hmean48']        = df['gh_hmean48']*df['early_ratio_smooth']
    df['adj_slot_mean']      = df['slot_mean48']*df['early_ratio_smooth']
    df['adj_gh_mean48']      = df['gh_mean48']*df['early_ratio_smooth']
    df['gh_q6mean48_scaled'] = df['gh_q6mean48']*df['early_ratio_smooth']
    df['gh_q2mean48_scaled'] = df['gh_q2mean48']*df['early_ratio_smooth']
    df['lag_rel']            = df['lag_d48']/(df['gh_hmean48']+1e-8)
    df['gh_hmean48_rel']     = df['gh_hmean48']/(df['gh_mean48']+1e-8)
    df['best_ratio']         = 0.4*df['d49_last_ratio'] + 0.6*df['early_ratio_smooth']
    df['best_adj_lag']       = df['lag_d48']*df['best_ratio']
    df['best_full_lag']      = df['best_adj_lag']*df['wx_demand_ratio']
    df['super_lag']          = df['full_adj_lag']*df['gh_wx_ratio']
    df['wx_change_temp']     = df['weather_changed']*df['temp_diff'].abs()
    df['time_wx']            = df['time_idx']*df['weather_changed']

print('All base features done.')

All base features done.


## Step 3 — GNN Message Passing (Core Innovation)

Two rounds of neighborhood aggregation:
- **1-hop**: aggregate features from direct spatial neighbors (~1 km)
- **2-hop**: aggregate the 1-hop aggregated features (2 km radius)

Key GNN features:
- `gnn_lag_*`: neighbor day-48 demand at same time slot → spatial demand surface
- `gnn_d49_*`: neighbor day-49 early demand → rescues sparse geohashes
- `gnn_ratio_*`: neighbor early_ratio → how busy is the neighborhood today?
- `gnn_full_adj_lag_*`: neighbor full corrected lag → best neighbor prediction

In [9]:
# ── Pre-compute per-geohash lookup tables for fast aggregation ────────────

# Number of neighbors per geohash (graph connectivity)
gh_n_neighbors = {g: len(adjacency[g]) for g in all_gh}

# Per-geohash scalar features (time-independent) for aggregation
gh_mean48_s     = dict(zip(d48.groupby('geohash')['demand'].mean().index,
                            d48.groupby('geohash')['demand'].mean().values))
gh_std48_s      = dict(zip(d48.groupby('geohash')['demand'].std().index,
                            d48.groupby('geohash')['demand'].std().values))

# Per-geohash early_ratio (scalar — same for all time slots of that geohash)
# Use the smoothed version from train/test (already computed)
# Build from d49_early only (not contaminated by training targets)
raw_ratio_map = (d49_mean_map / (d48_match_map + 1e-8)).clip(0.2, 5.0)
gh_ratio_for_gnn = raw_ratio_map.to_dict()  # geohash -> early_ratio

# Per-geohash d49 early mean (for GNN neighbor rescue)
gh_d49_mean_gnn = d49_mean_map.to_dict()

print('Pre-computed GNN lookup tables.')

Pre-computed GNN lookup tables.


In [10]:
# ── 1-hop message passing: time-slot-aware lag aggregation ───────────────
# For each (geohash, time_idx): aggregate neighbor day-48 demand at same slot

print('Computing GNN 1-hop lag features (slot-aware)...')
# Build (geohash, time_idx) → demand map from day-48
d48_gt_map = d48.set_index(['geohash','time_idx'])['demand'].to_dict()

def compute_gnn_lag_1hop(df, d48_gt_map, adjacency, global_val):
    means = np.full(len(df), np.nan)
    stds  = np.full(len(df), np.nan)
    maxs  = np.full(len(df), np.nan)
    for i, (gh, tidx) in enumerate(zip(df['geohash'].values, df['time_idx'].values)):
        nbrs = adjacency.get(gh, [])
        vals = [d48_gt_map.get((n, tidx), np.nan) for n in nbrs]
        vals = [v for v in vals if not np.isnan(v)]
        if vals:
            means[i] = np.mean(vals)
            stds[i]  = np.std(vals) if len(vals) > 1 else 0.0
            maxs[i]  = np.max(vals)
    return means, stds, maxs

for df in [train, test]:
    m, s, mx = compute_gnn_lag_1hop(df, d48_gt_map, adjacency, GLOBAL_MEAN)
    df['gnn_lag_mean_1h'] = m
    df['gnn_lag_std_1h']  = s
    df['gnn_lag_max_1h']  = mx
    # Fill missing (isolated nodes) with own lag
    df['gnn_lag_mean_1h'] = df['gnn_lag_mean_1h'].fillna(df['lag_d48'])
    df['gnn_lag_std_1h']  = df['gnn_lag_std_1h'].fillna(0.)
    df['gnn_lag_max_1h']  = df['gnn_lag_max_1h'].fillna(df['lag_d48'])

print('  gnn_lag_*_1h done.')

Computing GNN 1-hop lag features (slot-aware)...
  gnn_lag_*_1h done.


In [11]:
# ── 1-hop: scalar geohash features ───────────────────────────────────────
print('Computing GNN 1-hop scalar features...')

def agg_scalar_1hop(df, value_map, default, agg='mean'):
    """Aggregate a per-geohash scalar over 1-hop neighbors."""
    result = np.full(len(df), np.nan)
    for i, gh in enumerate(df['geohash'].values):
        nbrs = adjacency.get(gh, [])
        vals = [value_map.get(n, np.nan) for n in nbrs]
        vals = [v for v in vals if not np.isnan(v)]
        if vals:
            result[i] = np.mean(vals) if agg == 'mean' else np.std(vals)
    return result

for df in [train, test]:
    # Neighbor early_ratio (how busy is the neighborhood today vs yesterday?)
    df['gnn_ratio_mean_1h'] = agg_scalar_1hop(df, gh_ratio_for_gnn, GLOBAL_RATIO)
    df['gnn_ratio_mean_1h'] = df['gnn_ratio_mean_1h'].fillna(df['early_ratio'])

    # Neighbor day-49 early mean (rescues geohashes with no day-49 observations)
    df['gnn_d49_mean_1h'] = agg_scalar_1hop(df, gh_d49_mean_gnn, GLOBAL_D49M)
    df['gnn_d49_mean_1h'] = df['gnn_d49_mean_1h'].fillna(df['d49_mean'])

    # Neighbor day-48 overall mean demand (spatial demand level)
    df['gnn_gh_mean48_1h'] = agg_scalar_1hop(df, gh_mean48_s, GLOBAL_MEAN)
    df['gnn_gh_mean48_1h'] = df['gnn_gh_mean48_1h'].fillna(df['gh_mean48'])

    # Neighbor connectivity (how well-connected is this geohash's neighborhood?)
    df['gnn_n_neighbors'] = df['geohash'].map(gh_n_neighbors).fillna(0)

    # Neighbor full adjusted lag (spatial view of corrected prediction)
    # Build per-geohash mean of full_adj_lag
    tmp = df.groupby('geohash')['full_adj_lag'].mean().to_dict()
    df['gnn_full_adj_lag_1h'] = agg_scalar_1hop(df, tmp, GLOBAL_MEAN)
    df['gnn_full_adj_lag_1h'] = df['gnn_full_adj_lag_1h'].fillna(df['full_adj_lag'])

print('  gnn_*_1h scalar features done.')

Computing GNN 1-hop scalar features...
  gnn_*_1h scalar features done.


In [12]:
# ── 2-hop: aggregate neighbor-of-neighbor features ────────────────────────
# Use gnn_lag_mean_1h as the node feature, aggregate again over neighbors
print('Computing GNN 2-hop features...')

# Build per-geohash mean of the 1-hop aggregated lag
gh_gnn_lag_1h_map_tr = train.groupby('geohash')['gnn_lag_mean_1h'].mean().to_dict()
gh_gnn_lag_1h_map_te = test.groupby('geohash')['gnn_lag_mean_1h'].mean().to_dict()
gh_gnn_lag_1h_map    = {**gh_gnn_lag_1h_map_tr, **gh_gnn_lag_1h_map_te}

gh_gnn_ratio_1h_tr = train.groupby('geohash')['gnn_ratio_mean_1h'].mean().to_dict()
gh_gnn_ratio_1h_te = test.groupby('geohash')['gnn_ratio_mean_1h'].mean().to_dict()
gh_gnn_ratio_1h    = {**gh_gnn_ratio_1h_tr, **gh_gnn_ratio_1h_te}

for df in [train, test]:
    df['gnn_lag_mean_2h']   = agg_scalar_1hop(df, gh_gnn_lag_1h_map,  GLOBAL_MEAN)
    df['gnn_lag_mean_2h']   = df['gnn_lag_mean_2h'].fillna(df['gnn_lag_mean_1h'])
    df['gnn_ratio_mean_2h'] = agg_scalar_1hop(df, gh_gnn_ratio_1h, GLOBAL_RATIO)
    df['gnn_ratio_mean_2h'] = df['gnn_ratio_mean_2h'].fillna(df['gnn_ratio_mean_1h'])

print('  gnn_*_2h features done.')

Computing GNN 2-hop features...
  gnn_*_2h features done.


In [13]:
# ── Derived GNN features ──────────────────────────────────────────────────
for df in [train, test]:
    # How different is this node from its neighborhood?
    df['gnn_lag_diff_1h']   = df['lag_d48'] - df['gnn_lag_mean_1h']
    df['gnn_ratio_diff_1h'] = df['early_ratio'] - df['gnn_ratio_mean_1h']

    # GNN-corrected lag: own lag scaled by neighbor ratio signal
    df['gnn_adj_lag_1h']    = df['lag_d48'] * df['gnn_ratio_mean_1h']
    df['gnn_adj_lag_2h']    = df['lag_d48'] * df['gnn_ratio_mean_2h']

    # Spatial demand surface at this time slot
    df['gnn_spatial_surf']  = df['gnn_lag_mean_1h'] * df['early_ratio_smooth']

    # GNN-rescued d49 mean (blend own + neighbor if own is missing)
    has_own_d49 = df['geohash'].isin(d49_mean_map.index)
    df['gnn_d49_rescued'] = np.where(
        has_own_d49,
        df['d49_mean'],
        df['gnn_d49_mean_1h']  # borrow from neighbors if no own data
    )

print('Derived GNN features done.')
gnn_features = [
    'gnn_lag_mean_1h', 'gnn_lag_std_1h', 'gnn_lag_max_1h',
    'gnn_ratio_mean_1h', 'gnn_d49_mean_1h', 'gnn_gh_mean48_1h',
    'gnn_n_neighbors', 'gnn_full_adj_lag_1h',
    'gnn_lag_mean_2h', 'gnn_ratio_mean_2h',
    'gnn_lag_diff_1h', 'gnn_ratio_diff_1h',
    'gnn_adj_lag_1h', 'gnn_adj_lag_2h',
    'gnn_spatial_surf', 'gnn_d49_rescued',
]
print(f'Total GNN features: {len(gnn_features)}')

Derived GNN features done.
Total GNN features: 16


## Step 4 — Encode Categoricals & Assemble Feature Matrix

In [14]:
CAT_COLS = ['RoadType','LargeVehicles','Landmarks','Weather','weather_d48','gh4','gh5','gh6']
combined = pd.concat([train[CAT_COLS], test[CAT_COLS]], axis=0)
for col in CAT_COLS:
    le = LabelEncoder(); le.fit(combined[col].astype(str))
    train[col] = le.transform(train[col].astype(str))
    test[col]  = le.transform(test[col].astype(str))

BASE_FEATURES = [
    'hour','minute','time_idx','hour_sin','hour_cos','minute_sin','minute_cos',
    'lat','lon','gh4','gh5','gh6',
    'geohash_te','gh5_te','gh4_te','gh6_te',
    'RoadType','NumberofLanes','LargeVehicles','Landmarks',
    'Temperature','Weather','weather_d48','weather_changed','temp_d48','temp_diff',
    'wx_demand_ratio','wx_adj_lag','full_adj_lag','wx_change_temp','time_wx',
    'gh_wx_mean','gh_wx_d48_mean','gh_wx_ratio',
    'lag_d48','adjusted_lag','early_ratio','adjusted_lag_smooth','early_ratio_smooth',
    'd49_last_ratio','best_ratio','best_adj_lag','best_full_lag','super_lag',
    'd49_mean','d49_max','d49_std','d49_last','d49_slope',
    'gh_mean48','gh_max48','gh_std48','gh_med48','gh_q25_48','gh_q75_48',
    'gh_hmean48','slot_mean48','gh_q6mean48','gh_q2mean48',
    'adj_hmean48','adj_slot_mean','adj_gh_mean48',
    'gh_q6mean48_scaled','gh_q2mean48_scaled',
    'lag_rel','gh_hmean48_rel','gh_d48_peak_tidx','dist_to_peak',
    'nb_m4','nb_m2','nb_m1','nb_p1','nb_p2','nb_p4','lag_slope','lag_slope_broad',
]

FEATURES = BASE_FEATURES + gnn_features

X_train = train[FEATURES].values.astype(np.float32)
y_train = train['demand'].values.astype(np.float32)
X_test  = test[FEATURES].values.astype(np.float32)

assert not np.isnan(X_train).any(), f'NaN in X_train!'
assert not np.isnan(X_test).any(),  f'NaN in X_test!'

d49_mask       = (train['day'].values == 49)
sample_weights = np.where(d49_mask, 4.0, 1.0).astype(np.float32)
n_real         = len(X_train)

print(f'Total features: {len(FEATURES)} (base={len(BASE_FEATURES)}, gnn={len(gnn_features)})')
print(f'X_train: {X_train.shape}   X_test: {X_test.shape}')

Total features: 92 (base=76, gnn=16)
X_train: (77299, 92)   X_test: (41778, 92)


## Step 5 — Model Training (Pass 1)

In [15]:
kf = KFold(n_splits=5, shuffle=True, random_state=SEED)

lgb_params = dict(
    n_estimators=3000, learning_rate=0.02, max_depth=10,
    num_leaves=255, min_child_samples=10, subsample=0.8, colsample_bytree=0.7,
    reg_alpha=0.05, reg_lambda=0.1, random_state=SEED, n_jobs=-1, verbose=-1)

lgb_dart_params = dict(
    boosting_type='dart', n_estimators=2000, learning_rate=0.02, max_depth=10,
    num_leaves=255, min_child_samples=10, drop_rate=0.1, max_drop=50,
    subsample=0.8, colsample_bytree=0.7, reg_alpha=0.05, reg_lambda=0.1,
    random_state=SEED, n_jobs=-1, verbose=-1)

xgb_params = dict(
    n_estimators=3000, learning_rate=0.02, max_depth=9,
    subsample=0.8, colsample_bytree=0.7, reg_alpha=0.05, reg_lambda=1.0,
    tree_method='hist', random_state=SEED, n_jobs=-1,
    eval_metric='rmse', early_stopping_rounds=50, verbosity=0)

hgb_params = dict(
    max_iter=3000, learning_rate=0.025, max_depth=11,
    min_samples_leaf=10, l2_regularization=0.02, max_features=0.8,
    early_stopping=True, n_iter_no_change=60, random_state=SEED)

et_params = dict(
    n_estimators=800, max_depth=None, min_samples_leaf=5,
    max_features=0.55, n_jobs=-1, random_state=SEED)

print('Parameters set. Starting training...')

Parameters set. Starting training...


In [16]:
print('=== LightGBM-standard (GNN features) ===')
oof_lgb  = np.zeros(n_real); test_lgb = np.zeros(len(X_test))
for fold, (tr, val) in enumerate(kf.split(X_train)):
    m = lgb.LGBMRegressor(**lgb_params)
    m.fit(X_train[tr], y_train[tr], sample_weight=sample_weights[tr],
          eval_set=[(X_train[val], y_train[val])],
          callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(-1)])
    oof_lgb[val] = m.predict(X_train[val]); test_lgb += m.predict(X_test)/5
    print(f'  Fold {fold+1}  R²={r2_score(y_train[val], oof_lgb[val]):.4f}')
lgb_d49 = r2_score(y_train[d49_mask], oof_lgb[d49_mask])
print(f'LGB: overall={r2_score(y_train,oof_lgb):.4f}  day49={lgb_d49:.4f}')

=== LightGBM-standard (GNN features) ===
  Fold 1  R²=0.9916
  Fold 2  R²=0.9911
  Fold 3  R²=0.9923
  Fold 4  R²=0.9912
  Fold 5  R²=0.9923
LGB: overall=0.9917  day49=0.9644


In [17]:
print('=== LightGBM-DART (GNN features) ===')
oof_dart  = np.zeros(n_real); test_dart = np.zeros(len(X_test))
for fold, (tr, val) in enumerate(kf.split(X_train)):
    m = lgb.LGBMRegressor(**lgb_dart_params)
    m.fit(X_train[tr], y_train[tr], sample_weight=sample_weights[tr],
          eval_set=[(X_train[val], y_train[val])])
    oof_dart[val] = m.predict(X_train[val]); test_dart += m.predict(X_test)/5
    print(f'  Fold {fold+1}  R²={r2_score(y_train[val], oof_dart[val]):.4f}')
dart_d49 = r2_score(y_train[d49_mask], oof_dart[d49_mask])
print(f'DART: overall={r2_score(y_train,oof_dart):.4f}  day49={dart_d49:.4f}')

=== LightGBM-DART (GNN features) ===
  Fold 1  R²=0.9870
  Fold 2  R²=0.9862
  Fold 3  R²=0.9870
  Fold 4  R²=0.9858
  Fold 5  R²=0.9871
DART: overall=0.9866  day49=0.9614


In [18]:
print('=== XGBoost (GNN features) ===')
oof_xgb  = np.zeros(n_real); test_xgb = np.zeros(len(X_test))
for fold, (tr, val) in enumerate(kf.split(X_train)):
    m = xgb.XGBRegressor(**xgb_params)
    m.fit(X_train[tr], y_train[tr], sample_weight=sample_weights[tr],
          eval_set=[(X_train[val], y_train[val])], verbose=False)
    oof_xgb[val] = m.predict(X_train[val]); test_xgb += m.predict(X_test)/5
    print(f'  Fold {fold+1}  R²={r2_score(y_train[val], oof_xgb[val]):.4f}')
xgb_d49 = r2_score(y_train[d49_mask], oof_xgb[d49_mask])
print(f'XGB: overall={r2_score(y_train,oof_xgb):.4f}  day49={xgb_d49:.4f}')

=== XGBoost (GNN features) ===
  Fold 1  R²=0.9921
  Fold 2  R²=0.9908
  Fold 3  R²=0.9921
  Fold 4  R²=0.9910
  Fold 5  R²=0.9920
XGB: overall=0.9916  day49=0.9644


In [19]:
print('=== HistGradientBoosting (GNN features) ===')
oof_hgb  = np.zeros(n_real); test_hgb = np.zeros(len(X_test))
for fold, (tr, val) in enumerate(kf.split(X_train)):
    m = HistGradientBoostingRegressor(**hgb_params)
    m.fit(X_train[tr], y_train[tr], sample_weight=sample_weights[tr])
    oof_hgb[val] = m.predict(X_train[val]); test_hgb += m.predict(X_test)/5
    print(f'  Fold {fold+1}  R²={r2_score(y_train[val], oof_hgb[val]):.4f}')
hgb_d49 = r2_score(y_train[d49_mask], oof_hgb[d49_mask])
print(f'HGB: overall={r2_score(y_train,oof_hgb):.4f}  day49={hgb_d49:.4f}')

=== HistGradientBoosting (GNN features) ===
  Fold 1  R²=0.9914
  Fold 2  R²=0.9904
  Fold 3  R²=0.9921
  Fold 4  R²=0.9905
  Fold 5  R²=0.9918
HGB: overall=0.9912  day49=0.9653


In [20]:
print('=== ExtraTrees (GNN features) ===')
oof_et  = np.zeros(n_real); test_et = np.zeros(len(X_test))
for fold, (tr, val) in enumerate(kf.split(X_train)):
    m = ExtraTreesRegressor(**et_params)
    m.fit(X_train[tr], y_train[tr], sample_weight=sample_weights[tr])
    oof_et[val] = m.predict(X_train[val]); test_et += m.predict(X_test)/5
    print(f'  Fold {fold+1}  R²={r2_score(y_train[val], oof_et[val]):.4f}')
et_d49 = r2_score(y_train[d49_mask], oof_et[d49_mask])
print(f'ET: overall={r2_score(y_train,oof_et):.4f}  day49={et_d49:.4f}')

=== ExtraTrees (GNN features) ===
  Fold 1  R²=0.9875
  Fold 2  R²=0.9868
  Fold 3  R²=0.9883
  Fold 4  R²=0.9858
  Fold 5  R²=0.9881
ET: overall=0.9873  day49=0.9642


In [21]:
# ── Pass-1 Ridge meta-stacker ─────────────────────────────────────────────
names_v1      = ['LGB', 'DART', 'XGB', 'HGB', 'ET']
oofs_v1       = [oof_lgb, oof_dart, oof_xgb, oof_hgb, oof_et]
tests_v1      = [test_lgb, test_dart, test_xgb, test_hgb, test_et]
d49_scores_v1 = np.array([lgb_d49, dart_d49, xgb_d49, hgb_d49, et_d49])

print('\nPass-1 day-49 R²:')
for n, d in zip(names_v1, d49_scores_v1):
    print(f'  {n}: {d:.4f}')

# Ridge stacker fit on day-49 OOF (distribution closest to test)
ridge_v1 = Ridge(alpha=1.0)
ridge_v1.fit(np.column_stack(oofs_v1)[d49_mask], y_train[d49_mask])
oof_ridge1  = ridge_v1.predict(np.column_stack(oofs_v1))
test_ridge1 = ridge_v1.predict(np.column_stack(tests_v1))
ridge1_d49  = r2_score(y_train[d49_mask], np.clip(oof_ridge1[d49_mask], 0, 1))

w_v1 = d49_scores_v1 / d49_scores_v1.sum()
oof_v1_ens  = sum(w*p for w,p in zip(w_v1, oofs_v1))
test_v1_ens = sum(w*p for w,p in zip(w_v1, tests_v1))
ens_d49_v1  = r2_score(y_train[d49_mask], oof_v1_ens[d49_mask])

print(f'\nPass-1 weighted ensemble: day49={ens_d49_v1:.4f}')
print(f'Pass-1 Ridge stacker:     day49={ridge1_d49:.4f}')

# Best pass-1 signal for pseudo-labeling
best_p1_d49 = max(ridge1_d49, ens_d49_v1)
test_preds_v1 = np.clip(test_ridge1 if ridge1_d49 >= ens_d49_v1 else test_v1_ens, 0, 1)
print(f'Pseudo-label source: {"Ridge" if ridge1_d49 >= ens_d49_v1 else "Ensemble"} ({best_p1_d49:.4f})')


Pass-1 day-49 R²:
  LGB: 0.9644
  DART: 0.9614
  XGB: 0.9644
  HGB: 0.9653
  ET: 0.9642

Pass-1 weighted ensemble: day49=0.9660
Pass-1 Ridge stacker:     day49=0.9667
Pseudo-label source: Ridge (0.9667)


## Step 6 — Pseudo-Labeling Pass 2

In [22]:
PSEUDO_WEIGHT = 0.3
X_aug      = np.vstack([X_train, X_test]).astype(np.float32)
y_aug      = np.concatenate([y_train, test_preds_v1.astype(np.float32)])
w_aug      = np.concatenate([sample_weights,
                              np.full(len(X_test), PSEUDO_WEIGHT, dtype=np.float32)])
pseudo_idx = np.arange(n_real, len(X_aug))
print(f'Pseudo-label pass: {len(X_aug)} rows (real={n_real}, pseudo={len(X_test)})')

Pseudo-label pass: 119077 rows (real=77299, pseudo=41778)


In [23]:
print('=== LGB v2 (GNN + pseudo) ===')
oof_lgb2  = np.zeros(n_real); test_lgb2 = np.zeros(len(X_test))
for fold, (tr, val) in enumerate(kf.split(np.arange(n_real))):
    tr_aug = np.concatenate([tr, pseudo_idx])
    m = lgb.LGBMRegressor(**lgb_params)
    m.fit(X_aug[tr_aug], y_aug[tr_aug], sample_weight=w_aug[tr_aug],
          eval_set=[(X_aug[val], y_aug[val])],
          callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(-1)])
    oof_lgb2[val] = m.predict(X_aug[val]); test_lgb2 += m.predict(X_test)/5
    print(f'  Fold {fold+1}  R²={r2_score(y_train[val], oof_lgb2[val]):.4f}')
lgb2_d49 = r2_score(y_train[d49_mask], oof_lgb2[d49_mask])
print(f'LGB-v2: overall={r2_score(y_train,oof_lgb2):.4f}  day49={lgb2_d49:.4f}  (was {lgb_d49:.4f})')

=== LGB v2 (GNN + pseudo) ===
  Fold 1  R²=0.9928
  Fold 2  R²=0.9921
  Fold 3  R²=0.9935
  Fold 4  R²=0.9928
  Fold 5  R²=0.9930
LGB-v2: overall=0.9929  day49=0.9684  (was 0.9644)


In [24]:
print('=== DART v2 (GNN + pseudo) ===')
oof_dart2  = np.zeros(n_real); test_dart2 = np.zeros(len(X_test))
for fold, (tr, val) in enumerate(kf.split(np.arange(n_real))):
    tr_aug = np.concatenate([tr, pseudo_idx])
    m = lgb.LGBMRegressor(**lgb_dart_params)
    m.fit(X_aug[tr_aug], y_aug[tr_aug], sample_weight=w_aug[tr_aug],
          eval_set=[(X_aug[val], y_aug[val])])
    oof_dart2[val] = m.predict(X_aug[val]); test_dart2 += m.predict(X_test)/5
    print(f'  Fold {fold+1}  R²={r2_score(y_train[val], oof_dart2[val]):.4f}')
dart2_d49 = r2_score(y_train[d49_mask], oof_dart2[d49_mask])
print(f'DART-v2: overall={r2_score(y_train,oof_dart2):.4f}  day49={dart2_d49:.4f}  (was {dart_d49:.4f})')

=== DART v2 (GNN + pseudo) ===
  Fold 1  R²=0.9876
  Fold 2  R²=0.9871
  Fold 3  R²=0.9879
  Fold 4  R²=0.9868
  Fold 5  R²=0.9877
DART-v2: overall=0.9874  day49=0.9646  (was 0.9614)


In [25]:
print('=== XGB v2 (GNN + pseudo) ===')
oof_xgb2  = np.zeros(n_real); test_xgb2 = np.zeros(len(X_test))
for fold, (tr, val) in enumerate(kf.split(np.arange(n_real))):
    tr_aug = np.concatenate([tr, pseudo_idx])
    m = xgb.XGBRegressor(**xgb_params)
    m.fit(X_aug[tr_aug], y_aug[tr_aug], sample_weight=w_aug[tr_aug],
          eval_set=[(X_aug[val], y_aug[val])], verbose=False)
    oof_xgb2[val] = m.predict(X_aug[val]); test_xgb2 += m.predict(X_test)/5
    print(f'  Fold {fold+1}  R²={r2_score(y_train[val], oof_xgb2[val]):.4f}')
xgb2_d49 = r2_score(y_train[d49_mask], oof_xgb2[d49_mask])
print(f'XGB-v2: overall={r2_score(y_train,oof_xgb2):.4f}  day49={xgb2_d49:.4f}  (was {xgb_d49:.4f})')

=== XGB v2 (GNN + pseudo) ===
  Fold 1  R²=0.9928
  Fold 2  R²=0.9918
  Fold 3  R²=0.9926
  Fold 4  R²=0.9919
  Fold 5  R²=0.9926
XGB-v2: overall=0.9923  day49=0.9681  (was 0.9644)


In [26]:
print('=== HGB v2 (GNN + pseudo) ===')
oof_hgb2  = np.zeros(n_real); test_hgb2 = np.zeros(len(X_test))
for fold, (tr, val) in enumerate(kf.split(np.arange(n_real))):
    tr_aug = np.concatenate([tr, pseudo_idx])
    m = HistGradientBoostingRegressor(**hgb_params)
    m.fit(X_aug[tr_aug], y_aug[tr_aug], sample_weight=w_aug[tr_aug])
    oof_hgb2[val] = m.predict(X_aug[val]); test_hgb2 += m.predict(X_test)/5
    print(f'  Fold {fold+1}  R²={r2_score(y_train[val], oof_hgb2[val]):.4f}')
hgb2_d49 = r2_score(y_train[d49_mask], oof_hgb2[d49_mask])
print(f'HGB-v2: overall={r2_score(y_train,oof_hgb2):.4f}  day49={hgb2_d49:.4f}  (was {hgb_d49:.4f})')

=== HGB v2 (GNN + pseudo) ===
  Fold 1  R²=0.9925
  Fold 2  R²=0.9914
  Fold 3  R²=0.9927
  Fold 4  R²=0.9920
  Fold 5  R²=0.9922
HGB-v2: overall=0.9922  day49=0.9693  (was 0.9653)


In [27]:
print('=== ET v2 (GNN + pseudo) ===')
oof_et2  = np.zeros(n_real); test_et2 = np.zeros(len(X_test))
for fold, (tr, val) in enumerate(kf.split(np.arange(n_real))):
    tr_aug = np.concatenate([tr, pseudo_idx])
    m = ExtraTreesRegressor(**et_params)
    m.fit(X_aug[tr_aug], y_aug[tr_aug], sample_weight=w_aug[tr_aug])
    oof_et2[val] = m.predict(X_aug[val]); test_et2 += m.predict(X_test)/5
    print(f'  Fold {fold+1}  R²={r2_score(y_train[val], oof_et2[val]):.4f}')
et2_d49 = r2_score(y_train[d49_mask], oof_et2[d49_mask])
print(f'ET-v2: overall={r2_score(y_train,oof_et2):.4f}  day49={et2_d49:.4f}  (was {et_d49:.4f})')

=== ET v2 (GNN + pseudo) ===
  Fold 1  R²=0.9877
  Fold 2  R²=0.9869
  Fold 3  R²=0.9884
  Fold 4  R²=0.9860
  Fold 5  R²=0.9882
ET-v2: overall=0.9874  day49=0.9658  (was 0.9642)


## Step 7 — Final Ensemble & Submission

In [28]:
names_v2      = ['LGB-v2', 'DART-v2', 'XGB-v2', 'HGB-v2', 'ET-v2']
oofs_v2       = [oof_lgb2, oof_dart2, oof_xgb2, oof_hgb2, oof_et2]
tests_v2      = [test_lgb2, test_dart2, test_xgb2, test_hgb2, test_et2]
d49_scores_v2 = np.array([lgb2_d49, dart2_d49, xgb2_d49, hgb2_d49, et2_d49])

w_v2        = d49_scores_v2 / d49_scores_v2.sum()
oof_v2_ens  = sum(w*p for w, p in zip(w_v2, oofs_v2))
test_v2_ens = sum(w*p for w, p in zip(w_v2, tests_v2))
ens_d49_v2  = r2_score(y_train[d49_mask], oof_v2_ens[d49_mask])

wx1 = (train['weather_changed'].values == 1) & d49_mask
wx0 = (train['weather_changed'].values == 0) & d49_mask

print('=== FINAL RESULTS ===')
print(f'Pass-1 ensemble: day49={ens_d49_v1:.4f}')
print(f'Pass-2 ensemble: day49={ens_d49_v2:.4f}')

best_preds = np.clip(test_v2_ens, 0, 1)
print(f'\nBest: pass2_ens  day49={ens_d49_v2:.4f}')
print(f'  wx=1: R²={r2_score(y_train[wx1], np.clip(oof_v2_ens[wx1],0,1)):.4f}  ({wx1.sum()} rows)')
print(f'  wx=0: R²={r2_score(y_train[wx0], np.clip(oof_v2_ens[wx0],0,1)):.4f}  ({wx0.sum()} rows)')

submission = pd.DataFrame({'Index': test['Index'], 'demand': best_preds})
submission.to_csv('submission_gnn_pass2_ens.csv', index=False)
submission.to_csv('submission_gnn.csv', index=False)
print(f'\nsubmission_gnn_pass2_ens.csv saved  mean={best_preds.mean():.4f}')


=== FINAL RESULTS ===
Pass-1 ensemble:      day49=0.9660
Pass-1 Ridge:         day49=0.9667
Pass-2 ensemble:      day49=0.9688
Pass-2 Ridge:         day49=0.9696

Best: pass2_ridge  day49=0.9696
  wx=1: R²=0.9684  (5468 rows)
  wx=0: R²=0.9722  (2404 rows)

submission_gnn.csv saved  mean=0.1299
All variants saved: submission_gnn_*.csv


## Step 8 — GNN Feature Importance Analysis

In [29]:
# Train a quick LGB on full data to get feature importances
fi_model = lgb.LGBMRegressor(n_estimators=500, learning_rate=0.05, max_depth=8,
                              num_leaves=127, random_state=SEED, verbose=-1)
fi_model.fit(X_train, y_train, sample_weight=sample_weights)

fi = pd.DataFrame({'feature': FEATURES, 'importance': fi_model.feature_importances_})
fi = fi.sort_values('importance', ascending=False)

print('Top 20 features:')
print(fi.head(20).to_string(index=False))

print('\nGNN feature ranks:')
gnn_fi = fi[fi['feature'].str.startswith('gnn_')]
print(gnn_fi.to_string(index=False))

Top 20 features:
           feature  importance
            minute        2785
             nb_m1        2697
             nb_p1        2654
       gh_q2mean48        2465
gh_q2mean48_scaled        1842
           d49_std        1442
         d49_slope        1248
          d49_mean        1157
           lag_rel        1032
         lag_slope         813
           d49_max         791
         temp_diff         722
          d49_last         675
      dist_to_peak         672
       gh_wx_ratio         661
   lag_slope_broad         633
       Temperature         627
     adj_slot_mean         535
             nb_p2         521
   wx_demand_ratio         519

GNN feature ranks:
            feature  importance
    gnn_lag_diff_1h         355
    gnn_d49_rescued         355
     gnn_lag_std_1h         326
    gnn_lag_mean_1h         304
   gnn_spatial_surf         244
     gnn_lag_max_1h         242
     gnn_adj_lag_1h         236
  gnn_ratio_diff_1h         230
  gnn_ratio_mean_2h     

## Section 9 — Post-processing Fine-Tuning (no model training)

Two techniques applied to `submission_gnn_pass2_ens.csv`:
1. **EWM temporal smoothing** per geohash — model predictions have slot-to-slot noise; smoothing with warmup from the known day-49 slot-8 anchor reduces it
2. **Soft blend with `super_lag` formula** — `lag_d48 × early_ratio_smooth × wx_demand_ratio × gh_wx_ratio` gives a robust rule-based anchor

Both validated on day-49 OOF before applying to test.

In [ ]:
# ── Post-processing: validate every strategy on day-49 OOF, then apply ────

sub_raw = pd.read_csv('submission_gnn_pass2_ens.csv')

# ── Day-49 OOF context (uses notebook variables already in memory) ─────────
d49_oof             = train[d49_mask].copy().reset_index(drop=True)
d49_oof['pred']     = oof_v2_ens[d49_mask]
d49_oof['super_lag']= train[d49_mask]['super_lag'].values
baseline_r2         = r2_score(d49_oof['demand'], d49_oof['pred'])
print(f'Baseline day-49 OOF R²: {baseline_r2:.6f}')

# ── S1: EWM temporal smoothing (different spans) ──────────────────────────
d49_oof_s = d49_oof.sort_values(['geohash', 'time_idx']).copy()
ewm_scores = {}
for span in [2, 3, 5]:
    ewm_p = (d49_oof_s.groupby('geohash')['pred']
             .transform(lambda x: x.ewm(span=span, adjust=False).mean()))
    r2 = r2_score(d49_oof_s['demand'], ewm_p.clip(0, 1))
    ewm_scores[span] = r2
    print(f'  EWM span={span}: R²={r2:.6f}  ({r2-baseline_r2:+.6f})')

best_ewm_span = max(ewm_scores, key=ewm_scores.get)
best_ewm_r2   = ewm_scores[best_ewm_span]

# ── S2: Soft blend with super_lag formula (α = fraction of super_lag) ─────
blend_scores = {}
for alpha in [0.02, 0.05, 0.10, 0.15, 0.20]:
    bl = (1 - alpha) * d49_oof['pred'] + alpha * d49_oof['super_lag'].clip(0, 1)
    r2 = r2_score(d49_oof['demand'], bl.clip(0, 1))
    blend_scores[alpha] = r2
    print(f'  Blend α={alpha:.2f}: R²={r2:.6f}  ({r2-baseline_r2:+.6f})')

best_alpha = max(blend_scores, key=blend_scores.get)
best_blend_r2 = blend_scores[best_alpha]

# ── S3: EWM + blend combined ───────────────────────────────────────────────
ewm_p_best = (d49_oof_s.groupby('geohash')['pred']
              .transform(lambda x: x.ewm(span=best_ewm_span, adjust=False).mean()))
d49_oof_s['pred_ewm'] = ewm_p_best.values
d49_oof_sorted_reset  = d49_oof_s.reset_index(drop=True)
# merge super_lag back
sl_map = d49_oof[['geohash','time_idx','super_lag']].set_index(['geohash','time_idx'])['super_lag']
d49_oof_s['super_lag'] = d49_oof_s.set_index(['geohash','time_idx']).index.map(sl_map).values
comb_scores = {}
for alpha in [0.02, 0.05, 0.10]:
    comb = (1 - alpha) * d49_oof_s['pred_ewm'] + alpha * d49_oof_s['super_lag'].clip(0, 1)
    r2   = r2_score(d49_oof_s['demand'], comb.clip(0, 1))
    comb_scores[alpha] = r2
    print(f'  EWM(span={best_ewm_span})+blend α={alpha:.2f}: R²={r2:.6f}  ({r2-baseline_r2:+.6f})')

best_comb_alpha = max(comb_scores, key=comb_scores.get)
best_comb_r2    = comb_scores[best_comb_alpha]

print(f'\nSummary:')
print(f'  Baseline:               {baseline_r2:.6f}')
print(f'  Best EWM (span={best_ewm_span}):       {best_ewm_r2:.6f}  ({best_ewm_r2-baseline_r2:+.6f})')
print(f'  Best blend (α={best_alpha:.2f}):     {best_blend_r2:.6f}  ({best_blend_r2-baseline_r2:+.6f})')
print(f'  Best EWM+blend (α={best_comb_alpha:.2f}): {best_comb_r2:.6f}  ({best_comb_r2-baseline_r2:+.6f})')

In [ ]:
# ── Apply the best strategy to test predictions ───────────────────────────
# Determine winner based on OOF validation above
strategies = {
    'ewm':      best_ewm_r2,
    'blend':    best_blend_r2,
    'combined': best_comb_r2,
}
winner = max(strategies, key=strategies.get)
print(f'Best strategy on OOF: {winner}  R²={strategies[winner]:.6f}')

# Merge submission with test features
sub_test = test[['Index', 'geohash', 'time_idx', 'super_lag']].merge(sub_raw, on='Index')
sub_test = sub_test.sort_values(['geohash', 'time_idx']).copy()

# ── EWM with warmup from d49_last_map (actual slot-8 demand) ──────────────
# d49_last_map is already in memory from cell-10: slot-8 actual demand per geohash
GLOBAL_WARMUP = float(np.mean(list(d49_last_map.values())))
ewm_alpha = 2.0 / (best_ewm_span + 1)

ewm_preds_list = []
for gh, grp in sub_test.groupby('geohash', sort=False):
    warmup = d49_last_map.get(gh, GLOBAL_WARMUP)
    state  = warmup
    out    = []
    for v in grp['demand'].values:
        state = ewm_alpha * v + (1.0 - ewm_alpha) * state
        out.append(state)
    ewm_preds_list.append(pd.Series(out, index=grp.index))

sub_test['demand_ewm'] = pd.concat(ewm_preds_list)

# ── Apply chosen strategy ─────────────────────────────────────────────────
if winner == 'ewm':
    sub_test['demand_ft'] = sub_test['demand_ewm'].clip(0, 1)
elif winner == 'blend':
    sub_test['demand_ft'] = ((1 - best_alpha) * sub_test['demand']
                             + best_alpha * sub_test['super_lag'].clip(0, 1)).clip(0, 1)
else:  # combined
    sub_test['demand_ft'] = ((1 - best_comb_alpha) * sub_test['demand_ewm']
                             + best_comb_alpha * sub_test['super_lag'].clip(0, 1)).clip(0, 1)

# Restore original Index order
sub_out = sub_test.sort_values('Index')[['Index', 'demand_ft']].rename(
    columns={'demand_ft': 'demand'})

sub_out.to_csv('submission_gnn_pass2_ens_ft.csv', index=False)
print(f'Saved submission_gnn_pass2_ens_ft.csv')
print(f'  mean={sub_out.demand.mean():.4f}  std={sub_out.demand.std():.4f}')
print(f'  original mean={sub_raw.demand.mean():.4f}  std={sub_raw.demand.std():.4f}')
print(f'  max change: {(sub_out.demand.values - sub_raw.demand.values).abs().max():.4f}')

## Section 10 — Generate sol.csv (OOF-calibrated + smoothed predictions)

In [39]:
sub_raw = pd.read_csv('submission_gnn_pass2_ens.csv')

# ── Step A: per-geohash OOF bias correction ───────────────────────────────
# For each geohash compute: how much does the model over/under-predict vs actuals?
d49_oof_df             = train[d49_mask].copy().reset_index(drop=True)
d49_oof_df['oof_pred'] = oof_v2_ens[d49_mask]

# actual mean / oof mean per geohash → the direction and magnitude of model bias
gh_actual_mean = d49_oof_df.groupby('geohash')['demand'].mean()
gh_oof_mean    = d49_oof_df.groupby('geohash')['oof_pred'].mean()
gh_oof_count   = d49_oof_df.groupby('geohash')['oof_pred'].count()

# Raw calibration factor, clipped to ±43%
gh_calib_raw = (gh_actual_mean / (gh_oof_mean + 1e-8)).clip(0.70, 1.43)

# Credibility-weighted: blend toward 1.0 when few OOF samples available
K_CRED       = 6
gh_calib     = (gh_oof_count * gh_calib_raw + K_CRED) / (gh_oof_count + K_CRED)

print(f'Per-geohash calibration factors:')
print(f'  mean={gh_calib.mean():.4f}  median={gh_calib.median():.4f}  std={gh_calib.std():.4f}')
print(f'  geohashes needing upscale   (>1.05): {(gh_calib>1.05).sum()}')
print(f'  geohashes needing downscale (<0.95): {(gh_calib<0.95).sum()}')
print(f'  geohashes already accurate  (±5%):  {((gh_calib>=0.95)&(gh_calib<=1.05)).sum()}')

# OOF R² with calibration applied (shows upper-bound estimate of gain)
for beta in [0.3, 0.5, 0.7]:
    oof_cal = d49_oof_df['oof_pred'] * d49_oof_df['geohash'].map(gh_calib).fillna(1.0) ** beta
    r2 = r2_score(d49_oof_df['demand'], oof_cal.clip(0, 1))
    print(f'  OOF R² β={beta}: {r2:.6f}  (baseline={r2_score(d49_oof_df["demand"], d49_oof_df["oof_pred"]):.6f})')

Per-geohash calibration factors:
  mean=1.0000  median=1.0033  std=0.0353
  geohashes needing upscale   (>1.05): 56
  geohashes needing downscale (<0.95): 85
  geohashes already accurate  (±5%):  937
  OOF R² β=0.3: 0.969187  (baseline=0.968839)
  OOF R² β=0.5: 0.969382  (baseline=0.968839)
  OOF R² β=0.7: 0.969547  (baseline=0.968839)


In [ ]:
# ── Step B: Apply calibration + EWM to test, save as sol.csv ────────────
# β=0.7 gave best OOF uplift (0.969547 vs 0.968839 baseline)
BEST_BETA = 0.7

# Map calibration factor to each test row
sub_raw2 = pd.read_csv('submission_gnn_pass2_ens.csv')
sub_sol  = test[['Index', 'geohash', 'time_idx']].copy()
sub_sol   = sub_sol.merge(sub_raw2, on='Index')

# Apply per-geohash calibration
calib_factor = sub_sol['geohash'].map(gh_calib).fillna(1.0) ** BEST_BETA
sub_sol['demand_cal'] = (sub_sol['demand'] * calib_factor).clip(0, 1)

# EWM smoothing with slot-8 warmup (span=2 is mildest, safest)
EWM_SPAN     = 2
ewm_alpha_s  = 2.0 / (EWM_SPAN + 1)
GLOBAL_WARMUP = float(np.mean(list(d49_last_map.values())))

sub_sol = sub_sol.sort_values(['geohash', 'time_idx']).copy()
ewm_out = []
for gh, grp in sub_sol.groupby('geohash', sort=False):
    warmup = d49_last_map.get(gh, GLOBAL_WARMUP)
    state  = warmup
    out    = []
    for v in grp['demand_cal'].values:
        state = ewm_alpha_s * v + (1.0 - ewm_alpha_s) * state
        out.append(state)
    ewm_out.append(pd.Series(out, index=grp.index))

sub_sol['demand_final'] = pd.concat(ewm_out).clip(0, 1)

# Restore original index order and save
sol_out = sub_sol.sort_values('Index')[['Index', 'demand_final']].rename(
    columns={'demand_final': 'demand'})
sol_out.to_csv('sol.csv', index=False)

# Compare stats
orig  = sub_raw2['demand'].values
final = sol_out['demand'].values
print('=== sol.csv saved ===')
print(f'  Rows:  {len(sol_out)}')
print(f'  mean:  {final.mean():.4f}  (was {orig.mean():.4f})')
print(f'  std:   {final.std():.4f}  (was {orig.std():.4f})')
print(f'  min:   {final.min():.4f}  (was {orig.min():.4f})')
print(f'  max:   {final.max():.4f}  (was {orig.max():.4f})')
print(f'  max change: {np.abs(final - orig).max():.4f}')
print(f'  mean change: {np.abs(final - orig).mean():.4f}')
print(f'  OOF R² after calibration (β=0.7): 0.969547  (baseline: 0.968839)')
